# 03 - Structured Cleaned Extract

In [1]:
import json
import re
 
from pathlib import Path

## 1. Konfigurasi

In [2]:
PIPELINE_VERSION = "v1.0"

ENABLE_TABLE_DETECTION = True
 
ENABLE_FIGURE_DETECTION = True

In [3]:
CLEAN_DIR      = Path("/kaggle/input/notebooks/muhammadnabilul/02-cleaning-extract/cleaned") 
STRUCTURED_DIR = Path("/kaggle/working/structured")
STRUCTURED_DIR.mkdir(parents=True, exist_ok=True,)
 
print(f"Input  : {CLEAN_DIR}")
print(f"Output : {STRUCTURED_DIR}")

Input  : /kaggle/input/notebooks/muhammadnabilul/02-cleaning-extract/cleaned
Output : /kaggle/working/structured


In [4]:
def load_json(path: Path):
    with open( path, encoding="utf-8") as f:
        return json.load(f)
        
def save_json(data, path: Path):
    with open(path,"w",encoding="utf-8") as f:
        json.dump(data,f,ensure_ascii=False,indent=2,)

In [5]:
json_files = []
 
for file in sorted(CLEAN_DIR.glob("*.json")):
    try:
        doc = load_json(file)
        if isinstance(doc, dict) and "pages" in doc:
            json_files.append(file)
            
    except Exception:
        continue
 
print(f"Dokumen valid: {len(json_files):,}")
 
for i, file in enumerate(json_files[:10], start=1):
    print(f"[{i:>2}] {file.name}")

Dokumen valid: 47
[ 1] analisis-petty-corruption-hasil-spak-2020---2024.json
[ 2] indeks-pembangunan-manusia-2022.json
[ 3] indeks-pembangunan-manusia-2023.json
[ 4] indeks-pembangunan-manusia-2024.json
[ 5] indeks-perilaku-anti-korupsi-2023.json
[ 6] indeks-perilaku-anti-korupsi-2024.json
[ 7] indikator-kesejahteraan-rakyat-2023.json
[ 8] indikator-kesejahteraan-rakyat-2024.json
[ 9] indikator-kesejahteraan-rakyat-2025.json
[10] indikator-sdgs-kesejahteraan-rakyat-2024.json


### Preview

In [6]:
if json_files:
 
    sample_doc = load_json(json_files[0])
    print(sample_doc["pages"][0]["text_clean"][:1500])
 
else:
    print("Tidak ada dokumen valid ditemukan.")

# WASIL SPAK D000 oat


## 2. Regex Pattern

In [7]:
MD_HEADING_PATTERN = re.compile(
    r'^(#{1,6})\s+(.+)$'
)
 
BAB_PATTERN = re.compile(
    r'^bab\s+([ivxlcdm]+|\d+)',
    re.IGNORECASE,
)
 
SECTION_PATTERN = re.compile(
    r'^(\d+(?:\.\d+)+)\s+(.+)$'
)
 
APPENDIX_PATTERN = re.compile(
    r'^lampiran\b',
    re.IGNORECASE,
)
 
LIST_SECTION_PATTERN = re.compile(
    r'^daftar\s+(isi|tabel|gambar|lampiran|grafik|peta)\b',
    re.IGNORECASE,
)
 
SPECIAL_SECTION_PATTERN = re.compile(
    r'^('
    r'kata pengantar'
    r'|abstrak'
    r'|abstract'
    r'|ringkasan eksekutif'
    r'|executive summary'
    r'|ringkasan'
    r'|penjelasan teknis'
    r'|penjelasan umum'
    r'|metodologi'
    r'|metode pengumpulan data'
    r'|konsep dan definisi'
    r'|concepts and definitions'
    r'|technical notes'
    r'|general notes'
    r'|ucapan terima kasih'
    r')$',
    re.IGNORECASE,
)
 
TABLE_PATTERN = re.compile(
    r'^tabel\s+(\d+(?:\.\d+)*)[\.:]?',
    re.IGNORECASE,
)

FIGURE_PATTERN = re.compile(
    r'^(gambar|grafik)\s+(\d+(?:\.\d+)*)[\.:]?',
    re.IGNORECASE,
)

## 3. Fungsi-fungsi

### a. Heading Detector

In [8]:
def detect_heading(line):
 
    line = line.strip()
 
    m = MD_HEADING_PATTERN.match(line)
    if m:
        return {
            "title": m.group(2).strip(),
            "level": len(m.group(1)),
            "source": "markdown",
        }
        
    m = BAB_PATTERN.match(line)
    if m:
        return {
            "title": line,
            "level": 1,
            "section_id": m.group(1),
            "source": "bab",
        }
     
    m = SECTION_PATTERN.match(line)
    if m:
        section_id = m.group(1)
        level = section_id.count(".") + 1
        return {
            "title": m.group(2).strip(),
            "level": level,
            "section_id": section_id,
            "source": "section",
        }
 
    if APPENDIX_PATTERN.match(line):
        return {
            "title": line,
            "level": 1,
            "section_id": "lampiran",
            "source": "lampiran",
        }
 
    if LIST_SECTION_PATTERN.match(line):
        return {
            "title": line,
            "level": 1,
            "source": "daftar",
        }
 
    if SPECIAL_SECTION_PATTERN.match(line):
        return {
            "title": line,
            "level": 1,
            "source": "special",
        }
 
    return None

### b. Table Detection

In [9]:
def detect_table_caption(line):
 
    if not ENABLE_TABLE_DETECTION:
        return None
 
    m = TABLE_PATTERN.match(line.strip())
 
    if not m:
        return None
 
    return {
        "table_id": m.group(1),
        "caption": line.strip(),
    }

### c. Figure Detection

In [10]:
def detect_figure_caption(line):
 
    if not ENABLE_FIGURE_DETECTION:
        return None
 
    m = FIGURE_PATTERN.match(line.strip())
 
    if not m:
        return None
 
    return {
        "figure_type": m.group(1).lower(),
        "figure_id": m.group(2),
        "caption": line.strip(),
    }

### d. Document Structuring

In [11]:
def structure_document(doc):
 
    pages    = doc.get("pages", [])
    metadata = doc.get("metadata", {})
 
    sections         = []
    document_tables  = []
    document_figures = []
 
    current_section = {
        "title":      "Prolog",
        "level":      0,
        "source":     "prolog",
        "page_start": None,
        "page_end":   None,
        "blocks":     [],
        "tables":     [],
        "figures":    [],
    }
 
    sections.append(current_section)
 
    for page in pages:
 
        page_number = page.get("page_number", 0)
        chunk_id    = page.get("chunk_id", "")
        text        = page.get("text_clean", "")
 
        if current_section["page_start"] is None:
            current_section["page_start"] = page_number
 
        lines       = text.splitlines()
        block_lines = []
 
        for line in lines:
 
            line = line.strip()
 
            if not line:
                continue
 
            heading = detect_heading(line)
 
            if heading:
 
                content = "\n".join(block_lines).strip()
                if content:
                    current_section["blocks"].append({
                        "page_number": page_number,
                        "chunk_id":    chunk_id,
                        "text":        content,
                    })
                block_lines = []
 
                current_section["page_end"] = page_number
 
                current_section = {
                    "title":      heading["title"],
                    "level":      heading["level"],
                    "source":     heading["source"],
                    "page_start": page_number,
                    "page_end":   None,
                    "blocks":     [],
                    "tables":     [],
                    "figures":    [],
                }
 
                if "section_id" in heading:
                    current_section["section_id"] = heading["section_id"]
 
                sections.append(current_section)
                continue
 
            table = detect_table_caption(line)
 
            if table:
                table.update({
                    "page_number": page_number,
                    "chunk_id":    chunk_id,
                    "section":     current_section["title"],
                })
                document_tables.append(table)
                current_section["tables"].append(table)
                block_lines.append(line)
                continue
 
            figure = detect_figure_caption(line)
 
            if figure:
                figure.update({
                    "page_number": page_number,
                    "chunk_id":    chunk_id,
                    "section":     current_section["title"],
                })
                document_figures.append(figure)
                current_section["figures"].append(figure)
                block_lines.append(line)
                continue
 
            block_lines.append(line)
 
        content = "\n".join(block_lines).strip()
        if content:
            current_section["blocks"].append({
                "page_number": page_number,
                "chunk_id":    chunk_id,
                "text":        content,
            })
 
        current_section["page_end"] = page_number

    sections = [
        s
        for s in sections
        if (
            s["blocks"]
            or s["tables"]
            or s["figures"]
        )
    ]
    
    return {
        "document_metadata": {
            "judul":     metadata.get("judul"),
            "tahun":     metadata.get("tahun"),
            "bps_url":   metadata.get("bps_url"),
            "nama_file": metadata.get("nama_file"),
        },
        "pipeline_version":  PIPELINE_VERSION,
        "sections":          sections,
        "document_tables":   document_tables,
        "document_figures":  document_figures,
    }

## 4. Main Program

### a. Process File

In [12]:
def process_file(input_path, output_dir):
    doc = load_json(input_path)
    structured = structure_document(doc)
    output_path = output_dir / input_path.name
    save_json(structured, output_path)
 
    return output_path

### b. Process Directory

In [13]:
def process_directory(json_files, output_dir):
 
    results = []
 
    for i, file in enumerate(json_files, start=1):
        print(f"[{i}/{len(json_files)}] {file.name}")
        out = process_file(file, output_dir)
        results.append(out)
 
    return results

In [14]:
results = process_directory(json_files, STRUCTURED_DIR)
 
print(f"\nSelesai memproses {len(results):,} dokumen.")

[1/47] analisis-petty-corruption-hasil-spak-2020---2024.json
[2/47] indeks-pembangunan-manusia-2022.json
[3/47] indeks-pembangunan-manusia-2023.json
[4/47] indeks-pembangunan-manusia-2024.json
[5/47] indeks-perilaku-anti-korupsi-2023.json
[6/47] indeks-perilaku-anti-korupsi-2024.json
[7/47] indikator-kesejahteraan-rakyat-2023.json
[8/47] indikator-kesejahteraan-rakyat-2024.json
[9/47] indikator-kesejahteraan-rakyat-2025.json
[10/47] indikator-sdgs-kesejahteraan-rakyat-2024.json
[11/47] indikator-sdgs-kesejahteraan-rakyat-2025.json
[12/47] infografis-statistik-kesejahteraan-rakyat-2025.json
[13/47] kesejahteraan-anak-indonesia--analisis-deprivasi-hak-anak-multidimensi-2022.json
[14/47] kesejahteraan-anak-indonesia--analisis-kemiskinan-anak-moneter-2022.json
[15/47] laporan-indeks-khusus-penanganan-stunting--ikps--2022-2023.json
[16/47] laporan-indeks-khusus-penanganan-stunting-2021-2022.json
[17/47] laporan-indeks-khusus-penanganan-stunting-kabupaten-kota-2021-2022.json
[18/47] profil-a